In [ ]:
#!/usr/bin/env python3

import os
import random
import tweepy
import re
import time
import logging
from datetime import datetime
from  import 
# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

class RashifalBot:
    def __init__(self):
        """Initialize bot with credentials from environment variables"""
        # Validate required environment variables
        required_vars = {
            'GROQ_KEY': 'Groq API key',
            'TWITTER_CONSUMER_KEY': 'Twitter Consumer Key',
            'TWITTER_CONSUMER_SECRET': 'Twitter Consumer Secret',
            'TWITTER_ACCESS_TOKEN': 'Twitter Access Token',
            'TWITTER_ACCESS_TOKEN_SECRET': 'Twitter Access Token Secret'
        }
        
        missing = []
        for var, description in required_vars.items():
            if not os.environ.get(var):
                missing.append(f"{var} ({description})")
        
        if missing:
            raise ValueError(f"Missing environment variables:\n" + "\n".join(missing))
        
        # Groq API
        groq_key = os.environ.get('GROQ_KEY')
        self.client = Groq(api_key=groq_key)
        self.groq_model = os.environ.get('GROQ_MODEL', 'llama-3.3-70b-versatile')
        logger.info(f"Groq initialized with model: {self.groq_model}")
        
        # Twitter credentials
        self.twitter_client = None
        self.twitter_api_v1 = None
        self.use_v1_api = False
        self.setup_twitter()
        
        # Zodiac signs with romanization
        self.zodiac_signs = [
            {"nepali": "मेष", "romanized": "Meṣa", "english": "Aries"},
            {"nepali": "वृषभ", "romanized": "Vṛṣabha", "english": "Taurus"},
            {"nepali": "मिथुन", "romanized": "Mithuna", "english": "Gemini"},
            {"nepali": "कर्कट", "romanized": "Karkaṭa", "english": "Cancer"},
            {"nepali": "सिंह", "romanized": "Siṃha", "english": "Leo"},
            {"nepali": "कन्या", "romanized": "Kanyā", "english": "Virgo"},
            {"nepali": "तुला", "romanized": "Tulā", "english": "Libra"},
            {"nepali": "वृश्चिक", "romanized": "Vṛśchika", "english": "Scorpio"},
            {"nepali": "धनु", "romanized": "Dhanu", "english": "Sagittarius"},
            {"nepali": "मकर", "romanized": "Makara", "english": "Capricorn"},
            {"nepali": "कुम्भ", "romanized": "Kumbha", "english": "Aquarius"},
            {"nepali": "मीन", "romanized": "Mīna", "english": "Pisces"}
        ]
        
        # Sign personality traits
        self.personalities = {
            "Aries": "impulsive trailblazer, energetic and enthusiastic, drags people on adventures, makes hasty decisions, natural leader who thrives on challenges",
            "Taurus": "stubborn bull, loves comfort and luxury, incredibly reliable but resistant to change, appreciates good food and beauty",
            "Gemini": "social butterfly with the twins, quick-witted and charming, talks to everyone, versatile but inconsistent, indecisive",
            "Cancer": "emotional crab, deeply intuitive nurturer, connected to home and family, moody and sensitive, offers shoulder to cry on",
            "Leo": "confident lion, natural performer who loves spotlight, generous and warm-hearted protector, can seem arrogant, undeniably loyal",
            "Virgo": "precise virgin, analytical and detail-oriented, strives for perfection, overly critical but wants to help, meticulous attention to everything",
            "Libra": "diplomatic scales, sees both sides, values fairness and beauty, thrives in artistic environments, quest for balance leads to indecisiveness",
            "Scorpio": "intense scorpion, passionate and magnetic, deeply loyal but secretive, vengeful if crossed, transforms and rises from challenges",
            "Sagittarius": "philosophical archer, optimistic freedom-lover, seeks knowledge and experiences, straightforward honesty mistaken for tactlessness",
            "Capricorn": "ambitious goat, hardworking and disciplined, achieves success through perseverance, serious and stern, strong sense of responsibility",
            "Aquarius": "innovative water bearer, forward-thinking humanitarian, independent and values freedom, unconventional ideas seem eccentric, visionary",
            "Pisces": "dreamy fish, intuitive and compassionate, creative and imaginative, sensitive leading to escapism, boundless empathy and kindness"
        }
    
    def setup_twitter(self):
        """Setup Twitter API from environment variables"""
        try:
            # Try API v2 first
            bearer_token = os.environ.get('TWITTER_BEARER_TOKEN')
            
            self.twitter_client = tweepy.Client(
                bearer_token=bearer_token if bearer_token else None,
                consumer_key=os.environ.get('TWITTER_CONSUMER_KEY'),
                consumer_secret=os.environ.get('TWITTER_CONSUMER_SECRET'),
                access_token=os.environ.get('TWITTER_ACCESS_TOKEN'),
                access_token_secret=os.environ.get('TWITTER_ACCESS_TOKEN_SECRET'),
                wait_on_rate_limit=True
            )
            
            try:
                me = self.twitter_client.get_me()
                logger.info(f"Twitter API v2 connected as @{me.data.username}")
                return
            except tweepy.errors.Forbidden as e:
                logger.warning(f"Twitter API v2 forbidden (might need elevated access): {e}")
                raise
            except Exception as e:
                logger.warning(f"Twitter API v2 failed: {e}, trying v1.1...")
                
                # Fallback to API v1.1
                auth = tweepy.OAuth1UserHandler(
                    os.environ.get('TWITTER_CONSUMER_KEY'),
                    os.environ.get('TWITTER_CONSUMER_SECRET'),
                    os.environ.get('TWITTER_ACCESS_TOKEN'),
                    os.environ.get('TWITTER_ACCESS_TOKEN_SECRET')
                )
                self.twitter_api_v1 = tweepy.API(auth, wait_on_rate_limit=True)
                user = self.twitter_api_v1.verify_credentials()
                logger.info(f"Twitter API v1.1 connected as @{user.screen_name}")
                self.use_v1_api = True
                
        except Exception as e:
            logger.error(f"Twitter setup failed: {e}")
            raise
    
    def clean_ai_text(self, text):
        """Remove AI giveaways and English zodiac names"""
        # Replace em-dashes and en-dashes with commas
        text = text.replace('—', ',').replace('–', ',')
        
        # Replace spaced dashes with commas
        text = re.sub(r'\s+-\s+', ', ', text)
        
        # Replace English zodiac names with romanized ones
        english_to_nepali = {
            'Aries': 'Meṣa',
            'Taurus': 'Vṛṣabha',
            'Gemini': 'Mithuna',
            'Cancer': 'Karkaṭa',
            'Leo': 'Siṃha',
            'Virgo': 'Kanyā',
            'Libra': 'Tulā',
            'Scorpio': 'Vṛśchika',
            'Sagittarius': 'Dhanu',
            'Capricorn': 'Makara',
            'Aquarius': 'Kumbha',
            'Pisces': 'Mīna'
        }
        
        # Replace English names (case insensitive, but preserve case of replacement)
        for english, nepali in english_to_nepali.items():
            # Replace possessive forms
            text = re.sub(f"{english}'s", f"{nepali}'s", text, flags=re.IGNORECASE)
            text = re.sub(f"\\b{english}\\b", nepali, text, flags=re.IGNORECASE)
        
        # Remove phrases like "as a [sign]"
        all_nepali_names = '|'.join(english_to_nepali.values())
        text = re.sub(f'\\bas a ({all_nepali_names})\\b', '', text, flags=re.IGNORECASE)
        text = re.sub(r',?\s*as a \w+,?\s*', ' ', text, flags=re.IGNORECASE)
        
        # Remove instruction lines
        lines = text.split('\n')
        cleaned_lines = []
        for line in lines:
            line = line.strip()
            if any(keyword in line.lower() for keyword in [
                'must be', 'should be', 'critical', 'mandatory', 'required',
                'strict', 'rule', 'format:', 'example:', 'write for',
                'now write', 'the horoscope:', 'message:', 'advice:'
            ]):
                continue
            if line:
                cleaned_lines.append(line)
        
        if cleaned_lines:
            text = cleaned_lines[0]
        
        # Remove meta-commentary
        meta_patterns = [
            r'^(A sentence like|Something like|Could be|For example|Like this|Try this|How about):\s*',
            r'^(So|Could be|For example|Like this|Something like)\b[,:]?\s*',
            r'^-\s*',
        ]
        
        for pattern in meta_patterns:
            text = re.sub(pattern, '', text, flags=re.IGNORECASE)
        
        # Remove quotes
        text = re.sub(r'"([^"]*)"', r'\1', text)
        
        # Remove AI phrases
        ai_phrases = [
            'as an AI', 'I cannot', 'I apologize', 'I understand',
            'must be a complete sentence', 'ending properly'
        ]
        
        for phrase in ai_phrases:
            text = text.replace(phrase, '').replace(phrase.capitalize(), '')
        
        # Clean up whitespace
        text = ' '.join(text.split())
        text = text.strip(' .,;:-')
        
        return text
    
    def generate_rashifal(self, sign_info, max_retries=3):
        """Generate rashifal for a sign with retry logic"""
        
        # Get sign personality traits
        personality = self.personalities[sign_info['english']]
        
        # Pick random other sign for relational horoscopes
        other_signs = [s for s in self.zodiac_signs if s['english'] != sign_info['english']]
        other_sign = random.choice(other_signs)
        
        # Randomly choose positive or negative tone (10% positive, 90% negative)
        tone = "positive" if random.random() < 0.1 else "negative"
        
        if tone == "positive":
            prompt = f"""Write ONE witty, slightly uplifting horoscope for {sign_info['romanized']}.

Examples:
- "{sign_info['romanized']}, your overthinking is finally paying off."
- "{sign_info['romanized']}, someone finally appreciates your intensity."
- "Your instincts were right all along, {sign_info['romanized']}."

Keep it natural length (10-20 words). Be encouraging but still witty.

NEVER use English zodiac names (Aries, Taurus, Gemini, Cancer, Leo, Virgo, Libra, Scorpio, Sagittarius, Capricorn, Aquarius, Pisces).
ONLY use romanized Nepali names: Meṣa, Vṛṣabha, Mithuna, Karkaṭa, Siṃha, Kanyā, Tulā, Vṛśchika, Dhanu, Makara, Kumbha, Mīna

Do NOT say "as a Leo" or "Libra's" - say "as a Siṃha" or "Tulā's".

Write for {sign_info['romanized']}:"""
        else:
            prompt = f"""Write ONE witty, snarky, brutally honest horoscope for {sign_info['romanized']}.

Examples of SNARKY, MEAN style:
- "{sign_info['romanized']}, pretending you don't care is getting exhausting, isn't it?"
- "Love isn't dead, {sign_info['romanized']}; it's just ignoring your texts."
- "You weren't ghosted, {sign_info['romanized']}; you were spiritually redirected."
- "{sign_info['romanized']}, stop calling it a 'situationship'; it's a warning sign."
- "They didn't change, {sign_info['romanized']}; you just stopped making excuses."
- "{sign_info['romanized']}, healing isn't an aesthetic."
- "You call it intuition, {sign_info['romanized']}; everyone else calls it paranoia."
- "Your peace is fragile, {sign_info['romanized']}; handle with caffeine."
- "{sign_info['romanized']}, stop calling self-sabotage 'a vibe.'"
- "The universe isn't testing you, {sign_info['romanized']}; you just refuse to learn."
- "Mercury isn't in retrograde, {sign_info['romanized']}; you just made bad choices."
- "The energy's off because you are, {sign_info['romanized']}."
- "{sign_info['romanized']}, stop refreshing their story; Wi-Fi pani thakcha."
- "{sign_info['romanized']}, maybe stop texting your ex every time it rains."
- "{sign_info['romanized']}, you're not manifesting; you're just procrastinating with incense."
- "{sign_info['romanized']}, they didn't ghost you; it's just load-shedding of emotions."
- "You can't vibe your way out of consequences, {sign_info['romanized']}."
- "{sign_info['romanized']}, your karmic debt looks like an eSewa balance right now; zero."
- "{sign_info['romanized']}, don't blame Mercury; blame that 2 a.m. call to your ex."
- "{sign_info['romanized']}, your aura looks like traffic near Kalanki right now."

Be MEAN, SNARKY, call them out. Natural length (15-30 words).

CRITICAL RULES:
- NEVER use English zodiac names (Aries, Taurus, Gemini, Cancer, Leo, Virgo, Libra, Scorpio, Sagittarius, Capricorn, Aquarius, Pisces)
- ONLY use romanized Nepali names at the START: Meṣa, Vṛṣabha, Mithuna, Karkaṭa, Siṃha, Kanyā, Tulā, Vṛśchika, Dhanu, Makara, Kumbha, Mīna
- Do NOT repeat the sign name twice in one horoscope
- Sign names should ONLY appear at the beginning of the horoscope

Write for {sign_info['romanized']}:"""
        
        retry_count = 0
        backoff_time = 60
        
        while retry_count < max_retries:
            try:
                completion = self.client.chat.completions.create(
                    model=self.groq_model,
                    messages=[
                        {
                            "role": "system",
                            "content": f"You write {'slightly uplifting' if tone == 'positive' else 'brutally honest, mean, snarky'} horoscopes. NEVER use English zodiac names like Leo, Libra, Aries. ONLY use romanized Nepali names. Be witty."
                        },
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    max_tokens=80,
                    temperature=0.9
                )
                
                # If we got here, API call succeeded
                logger.info(f"Groq API call successful (attempt {retry_count + 1})")
                break
                
            except Exception as api_error:
                retry_count += 1
                error_str = str(api_error).lower()
                
                if any(x in error_str for x in ["rate", "429", "limit", "too many"]):
                    if retry_count < max_retries:
                        wait_time = backoff_time * retry_count
                        logger.warning(f"Rate limit hit, attempt {retry_count}/{max_retries}")
                        logger.info(f"Waiting {wait_time} seconds before retry...")
                        time.sleep(wait_time)
                    else:
                        logger.error(f"Rate limit exceeded after {max_retries} attempts")
                        raise Exception("Rate limit exceeded, cannot generate horoscope")
                else:
                    logger.error(f"Groq API error: {api_error}")
                    raise api_error
        
        try:
            if completion and completion.choices:
                message = completion.choices[0].message
                
                if message and hasattr(message, 'content') and message.content:
                    raw_text = message.content.strip()
                    
                    # Check for meta-commentary
                    if re.match(r'^(A sentence like|Something like|Could be|For example|Like this|Try this|How about):', raw_text, re.IGNORECASE):
                        match = re.search(r'[:"]\s*(.+?)[".]?\s*$', raw_text)
                        rashifal_text = match.group(1).strip() if match else None
                    else:
                        rashifal_text = raw_text
                    
                    if rashifal_text:
                        rashifal_text = self.clean_ai_text(rashifal_text)
                        
                        # Add period if missing
                        if rashifal_text and not rashifal_text.endswith(('.', '!', '?')):
                            rashifal_text = rashifal_text.rstrip(',') + '.'
                    
                    if not rashifal_text:
                        raise Exception("Empty response after cleaning")
                    
                    logger.info(f"Raw generated: {raw_text}")
                    logger.info(f"Cleaned: {rashifal_text}")
                    logger.info(f"Tone: {tone}")
                    return rashifal_text
            
            # If generation fails completely, raise error
            raise Exception("Failed to generate valid horoscope")
            
        except Exception as e:
            logger.error(f"Generation error: {e}")
            raise
    
    def format_tweet(self, rashifal, sign_info):
        """Format and validate tweet text"""
        # Clean the generated text
        rashifal = self.clean_ai_text(rashifal)
        
        # If it already starts with the sign name, use it as is
        if rashifal.startswith(sign_info['romanized']):
            tweet_text = rashifal
        else:
            # Otherwise, add the sign name at the beginning
            # Capitalize first letter of message
            if rashifal:
                rashifal = rashifal[0].upper() + rashifal[1:]
            tweet_text = f"{sign_info['romanized']}, {rashifal}"
        
        # Enforce Twitter character limit (280)
        if len(tweet_text) > 280:
            logger.warning(f"Tweet too long ({len(tweet_text)} chars), truncating...")
            tweet_text = tweet_text[:277] + "..."
        
        return tweet_text
    
    def post_tweet(self, rashifal, sign_info, max_retries=3):
        """Post rashifal to Twitter with retry logic"""
        tweet_text = self.format_tweet(rashifal, sign_info)
        
        retry_count = 0
        backoff_time = 30
        
        while retry_count < max_retries:
            try:
                # Post
                if self.use_v1_api and self.twitter_api_v1:
                    response = self.twitter_api_v1.update_status(tweet_text)
                    tweet_id = response.id_str
                    tweet_url = f"https://twitter.com/i/web/status/{tweet_id}"
                else:
                    response = self.twitter_client.create_tweet(text=tweet_text)
                    tweet_id = response.data['id']
                    tweet_url = f"https://twitter.com/i/web/status/{tweet_id}"
                
                logger.info(f"Tweet posted successfully!")
                logger.info(f"Sign: {sign_info['romanized']} ({sign_info['english']})")
                logger.info(f"Tweet: {tweet_text}")
                logger.info(f"Tweet ID: {tweet_id}")
                logger.info(f"URL: {tweet_url}")
                logger.info(f"Characters: {len(tweet_text)}/280")
                
                return True
                
            except tweepy.errors.TooManyRequests as e:
                retry_count += 1
                if retry_count < max_retries:
                    wait_time = backoff_time * retry_count
                    logger.warning(f"Twitter rate limit hit, attempt {retry_count}/{max_retries}")
                    logger.info(f"Waiting {wait_time} seconds before retry...")
                    time.sleep(wait_time)
                else:
                    logger.error(f"Twitter rate limit exceeded after {max_retries} attempts")
                    logger.error(f"Failed tweet text: {tweet_text}")
                    return False
                    
            except tweepy.errors.Forbidden as e:
                logger.error(f"Twitter Forbidden error (check app permissions): {e}")
                logger.error(f"Failed tweet text: {tweet_text}")
                return False
                
            except Exception as e:
                logger.error(f"Tweet posting failed: {e}")
                logger.error(f"Failed tweet text: {tweet_text}")
                retry_count += 1
                if retry_count < max_retries:
                    wait_time = backoff_time * retry_count
                    logger.info(f"Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    return False
        
        return False

def main():
    """Main function to run the bot"""
    logger.info("=" * 60)
    logger.info("Starting Rashifal Twitter Bot")
    logger.info(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logger.info("=" * 60)
    
    try:
        # Initialize bot
        logger.info("\nInitializing bot...")
        bot = RashifalBot()
        
        # Pick random sign
        sign = random.choice(bot.zodiac_signs)
        logger.info(f"\nSelected sign: {sign['romanized']} ({sign['english']})")
        
        # Generate rashifal
        logger.info("Generating rashifal...")
        rashifal = bot.generate_rashifal(sign)
        logger.info(f"Generated: {rashifal}")
        
        # Post to Twitter
        logger.info("\nPosting to Twitter...")
        success = bot.post_tweet(rashifal, sign)
        
        if success:
            logger.info("\n" + "=" * 60)
            logger.info("SUCCESS! Bot run completed successfully")
            logger.info("=" * 60)
            return 0
        else:
            logger.error("\n" + "=" * 60)
            logger.error("FAILED: Could not post tweet")
            logger.error("=" * 60)
            return 1
            
    except Exception as e:
        logger.error("\n" + "=" * 60)
        logger.error(f"FATAL ERROR: {e}")
        logger.error("=" * 60)
        import traceback
        traceback.print_exc()
        return 1

if __name__ == "__main__":
    exit(main())